# Unichart: default per-dataset formatting (`set_default_format`)

`set_default_format(...)` sets the per-dataset **style defaults** applied to
datasets *as they are loaded* — the markersize / linestyle / linewidth / etc.
analogue of how `color_map` and `marker_map` assign colors and markers by index.

This notebook shows:

1. The built-in defaults a freshly loaded dataset gets.
2. Changing the defaults so **future** loaded datasets are restyled.
3. That **already-loaded** datasets are left untouched.
4. `reset_format()` re-applying the current defaults to existing datasets.
5. `set_default_format(reset=True)` restoring the built-ins.
6. `fill=False` hollow markers — working in both `by='vars'` and `by='sets'`.

Safe to *Run All* (synthetic data).

### Dependencies

`unichart` needs `scipy` and `ipywidgets` (and `kaleido` for static export).
On a managed system Python: `pip install --break-system-packages scipy ipywidgets kaleido`.

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))   # so `import unichart` resolves to this repo
import numpy as np
import pandas as pd
import unichart as uc

np.random.seed(7)

## 0. Data + a tiny helper to print a dataset's style

In [3]:
def make_run(drift=0.0, n=25):
    x = np.linspace(0, 10, n)
    return pd.DataFrame({
        'time':   x,
        'signal': np.cumsum(np.random.randn(n)) + drift * x,
        'temp':   20 + 3 * np.sin(x) + np.random.randn(n),
    })

def style_of(ds):
    return dict(markersize=ds.markersize, linestyle=ds.linestyle, linewidth=ds.linewidth,
                edgewidth=ds.edgewidth, edge_color=ds.edge_color, alpha=ds.alpha, fill=ds.fill)

---
## 1. Built-in defaults

A brand-new notebook loads datasets with the built-in style (markersize 10, no
line, linewidth 2, solid fill, ...).

In [4]:
nb = uc.UnichartNotebook()
nb.load_df(make_run(0.0), title='default_A')
nb.load_df(make_run(0.6), title='default_B')
print('built-in style of set A:', style_of(nb.sets[0]))
nb.plot(x='time', y='signal', suptitle='Built-in default styling', figsize=(9, 5))

UniChart Notebook Environment Initialized.
Loaded Set 0: default_A
Loaded Set 1: default_B
built-in style of set A: {'markersize': 10, 'linestyle': None, 'linewidth': 2, 'edgewidth': 1, 'edge_color': 'black', 'alpha': 1, 'fill': True}


---
## 2. Restyle *future* datasets

Change the defaults, then load two more datasets. Only the **new** datasets pick
up the change; the ones loaded earlier keep their styling.

In [5]:
nb.set_default_format(markersize=5, linestyle='--', linewidth=1.5)
nb.load_df(make_run(0.0),  title='new_C')
nb.load_df(make_run(-0.5), title='new_D')

print('set A loaded BEFORE the change (unchanged):', style_of(nb.sets[0]))
print('set C loaded AFTER  the change (restyled) :', style_of(nb.sets[2]))

Loaded Set 2: new_C
Loaded Set 3: new_D
set A loaded BEFORE the change (unchanged): {'markersize': 10, 'linestyle': None, 'linewidth': 2, 'edgewidth': 1, 'edge_color': 'black', 'alpha': 1, 'fill': True}
set C loaded AFTER  the change (restyled) : {'markersize': 5, 'linestyle': '--', 'linewidth': 1.5, 'edgewidth': 1, 'edge_color': 'black', 'alpha': 1, 'fill': True}


Overlaying all four shows it directly — A & B are big solid markers (old
defaults), C & D are small dashed lines (new defaults):

In [6]:
nb.plot(x='time', y='signal',
        suptitle='Old defaults (A, B) vs new defaults (C, D), overlaid',
        figsize=(10, 6))

---
## 3. Apply new defaults to existing datasets with `reset_format()`

Already-loaded datasets keep their look until you re-style them. `reset_format()`
resets datasets to the **current** defaults (not the built-ins) — so it pushes the
new defaults onto the older sets too.

In [7]:
nb.reset_format(vars=False, lines=False, highlights=False, scale=False, fonts=False)
print('set A after reset_format() -> current defaults:', style_of(nb.sets[0]))
nb.plot(x='time', y='signal',
        suptitle='After reset_format(): every set uses the current defaults',
        figsize=(10, 6))

Reset: dataset formatting.
set A after reset_format() -> current defaults: {'markersize': 5, 'linestyle': '--', 'linewidth': 1.5, 'edgewidth': 1, 'edge_color': 'black', 'alpha': 1, 'fill': True}


---
## 4. Restore the built-in defaults

`set_default_format(reset=True)` puts the defaults back to the originals for
future loads (existing datasets are not touched).

In [8]:
nb.set_default_format(reset=True)
nb.load_df(make_run(0.3), title='back_to_builtin')
print('newest set style:', style_of(nb.sets[-1]))

Default dataset format reset to built-ins.
Loaded Set 4: back_to_builtin
newest set style: {'markersize': 10, 'linestyle': None, 'linewidth': 2, 'edgewidth': 1, 'edge_color': 'black', 'alpha': 1, 'fill': True}


---
## 5. `fill=False` — hollow markers

`fill` toggles solid vs. hollow (transparent fill, colored outline) markers. This
now works in **both** plotting paths.

### 5a. `by='vars'` (datasets overlaid)

In [9]:
nbf = uc.UnichartNotebook()
nbf.set_default_format(markersize=14, fill=False, edgewidth=2)
for i in range(3):
    nbf.load_df(make_run(0.4 * i), title=f'run_{i}')

nbf.plot(x='time', y='signal',
         suptitle='Hollow markers (fill=False) — by="vars"', figsize=(9, 5))

UniChart Notebook Environment Initialized.
Loaded Set 0: run_0
Loaded Set 1: run_1
Loaded Set 2: run_2


### 5b. `by='sets'` (one subplot per dataset, incl. a secondary axis)

This is the path that previously ignored `fill`; hollow markers now render here
too, on both the primary and secondary y traces.

In [12]:
nbf.plot(x='time', y=['signal', 'temp'], by='sets',
         suptitle='Hollow markers (fill=False) — by="sets"', figsize=(10, 6))

---
## 6. Notes

* **Colors and markers** are still assigned by index via `color_map` / `marker_map`,
  not `set_default_format` (which covers markersize, linestyle, linewidth, edgewidth,
  edge_color, alpha, and fill).
* Values are validated up front — invalid settings raise immediately:

In [11]:
for kw in [dict(markersize=-1), dict(alpha=2), dict(linestyle='zigzag'), dict(fill='maybe')]:
    try:
        nbf.set_default_format(**kw)
        print('accepted (unexpected):', kw)
    except (ValueError, TypeError) as e:
        print(f'{kw}  ->  {type(e).__name__}: {e}')

{'markersize': -1}  ->  ValueError: markersize must be >= 0.0, got -1
{'alpha': 2}  ->  ValueError: alpha must be in [0.0, 1.0], got 2
{'linestyle': 'zigzag'}  ->  ValueError: Invalid linestyle 'zigzag'. Valid: ,  , -, --, -., :, None
{'fill': 'maybe'}  ->  ValueError: Invalid value for fill: maybe
